# Bayesian Return Modeling

Estimates posterior distributions for asset returns and volatility using a
conjugate Normal-Inverse-Gamma model implemented with NumPy and SciPy.
No external Bayesian library is required.

In [1]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

OverflowError: cannot convert longdouble infinity to integer

In [ ]:
returns = pd.read_csv('../data/raw/stock_returns.csv', index_col='Date')
returns.index = pd.to_datetime(returns.index)
returns = returns.select_dtypes(include='number').dropna()
print(f'Shape: {returns.shape}')

## Conjugate Normal-Inverse-Gamma Posterior

Prior: $\mu_0, \kappa_0, \alpha_0, \beta_0$  
Likelihood: $x_i \sim \mathcal{N}(\mu, \sigma^2)$  
Posterior parameters are updated analytically.

In [ ]:
def nig_posterior(data, mu0=0.0, kappa0=1.0, alpha0=1.0, beta0=1e-4):
    """
    Compute Normal-Inverse-Gamma posterior parameters given observed data.

    Returns a dict with keys: mu_n, kappa_n, alpha_n, beta_n,
    posterior_mean, posterior_std, credible_interval_95.
    """
    x   = np.asarray(data, dtype=float)
    n   = len(x)
    x_bar = x.mean()
    s2    = x.var(ddof=1) if n > 1 else 0.0

    kappa_n = kappa0 + n
    mu_n    = (kappa0 * mu0 + n * x_bar) / kappa_n
    alpha_n = alpha0 + n / 2.0
    beta_n  = beta0 + 0.5 * (n - 1) * s2 + (kappa0 * n * (x_bar - mu0)**2) / (2.0 * kappa_n)

    # Posterior predictive: Student-t for the mean
    df      = 2.0 * alpha_n
    scale   = np.sqrt(beta_n * (kappa_n + 1) / (alpha_n * kappa_n))
    t_dist  = stats.t(df=df, loc=mu_n, scale=scale)
    ci_lo, ci_hi = t_dist.ppf(0.025), t_dist.ppf(0.975)

    return {
        'mu_n': mu_n,
        'kappa_n': kappa_n,
        'alpha_n': alpha_n,
        'beta_n': beta_n,
        'posterior_mean': mu_n,
        'posterior_var_mean': beta_n / (alpha_n - 1) if alpha_n > 1 else np.nan,
        'credible_interval_95': (ci_lo, ci_hi),
    }

In [ ]:
# Compute posterior for each ticker
rows = []
for col in returns.columns:
    p = nig_posterior(returns[col].values)
    rows.append({
        'ticker': col,
        'posterior_mean_daily': p['posterior_mean'],
        'posterior_mean_annual': p['posterior_mean'] * 252,
        'posterior_var': p['posterior_var_mean'],
        'ci_low_95': p['credible_interval_95'][0],
        'ci_high_95': p['credible_interval_95'][1],
    })

posterior_df = pd.DataFrame(rows).set_index('ticker')
print(posterior_df.round(6))

## Posterior Predictive Visualisation

In [ ]:
def plot_posterior_predictive(data, ticker, mu0=0.0, kappa0=1.0, alpha0=1.0, beta0=1e-4):
    p = nig_posterior(data, mu0, kappa0, alpha0, beta0)
    df    = 2.0 * p['alpha_n']
    scale = np.sqrt(p['beta_n'] * (p['kappa_n'] + 1) / (p['alpha_n'] * p['kappa_n']))
    t_dist = stats.t(df=df, loc=p['mu_n'], scale=scale)

    x_plot = np.linspace(t_dist.ppf(0.001), t_dist.ppf(0.999), 500)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(data, bins=60, density=True, alpha=0.45, color='steelblue', label='Empirical')
    ax.plot(x_plot, t_dist.pdf(x_plot), color='crimson', linewidth=2, label='Posterior predictive')
    ci = p['credible_interval_95']
    ax.axvline(ci[0], color='gray', linestyle='--', linewidth=1, label='95% CI')
    ax.axvline(ci[1], color='gray', linestyle='--', linewidth=1)
    ax.set_title(f'{ticker} – Posterior Predictive Distribution')
    ax.set_xlabel('Daily Return')
    ax.set_ylabel('Density')
    ax.legend()
    plt.tight_layout()
    return fig

In [ ]:
for ticker in list(returns.columns)[:4]:
    fig = plot_posterior_predictive(returns[ticker].values, ticker)
    plt.show()

## Bayesian Portfolio Mean Estimate

In [ ]:
# Shrinkage: blend sample mean toward a grand prior mean (James-Stein style)
grand_mean = returns.values.mean()
shrinkage  = 0.5

shrunk_means = shrinkage * returns.mean() + (1 - shrinkage) * grand_mean
print('Shrunk annualised return estimates:')
print((shrunk_means * 252).round(4))

In [ ]:
# Compare MLE vs Bayesian shrinkage annualised means
compare = pd.DataFrame({
    'MLE mean (ann)': returns.mean() * 252,
    'Bayesian shrinkage (ann)': shrunk_means * 252,
})

fig, ax = plt.subplots(figsize=(10, 5))
compare.plot(kind='bar', ax=ax)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('MLE vs Bayesian Shrinkage Mean Return Estimates')
ax.set_ylabel('Annualised Return')
ax.set_xlabel('Ticker')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()